# Step 2: General Preparation (Spark Port)

In [1]:
import sys
sys.path.insert(0, '/home/jovyan/work/src')

from pyspark.sql import functions as F
from spark_utils import get_spark_session, save_parquet
from general_prep import (
    add_int_by_extracting_number,
    add_months_since_column,
    fit_dummy_spec,
    apply_dummy_columns,
    get_missing_summary,
    fill_null_with_column,
    fit_mean_imputer,
    apply_mean_imputer,
    fill_null_with_zero,
    check_null_count,
    add_good_bad_target,
)

In [2]:
SAMPLE_MODE = False # True for quick development; False for full dataset

## 2A.1 Load Cleaned Base Data

In [3]:
spark = get_spark_session()
df = spark.read.parquet('/home/jovyan/work/data/parquet/accepted_cleaned.parquet')

if SAMPLE_MODE:
    df = df.sample(fraction=0.1, seed=42)

print(f'Rows: {df.count()}, Columns: {len(df.columns)}')

Spark driver memory: 7g
Rows: 2260701, Columns: 155


In [4]:
df

DataFrame[id: string, member_id: string, loan_amnt: double, funded_amnt: double, funded_amnt_inv: double, term: string, int_rate: double, installment: double, grade: string, sub_grade: string, emp_title: string, emp_length: string, home_ownership: string, annual_inc: string, verification_status: string, issue_d: string, loan_status: string, pymnt_plan: string, url: string, desc: string, purpose: string, title: string, zip_code: string, addr_state: string, dti: string, delinq_2yrs: string, earliest_cr_line: string, fico_range_low: string, fico_range_high: string, inq_last_6mths: string, mths_since_last_delinq: string, mths_since_last_record: string, open_acc: string, pub_rec: string, revol_bal: string, revol_util: string, total_acc: string, initial_list_status: string, out_prncp: string, out_prncp_inv: string, total_pymnt: string, total_pymnt_inv: string, total_rec_prncp: string, total_rec_int: string, total_rec_late_fee: string, recoveries: string, collection_recovery_fee: string, last

## 2A.2 Train/Test Split

In [5]:
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

print(f'Train rows: {train_df.count()}')
print(f'Test rows: {test_df.count()}')

Train rows: 1809174
Test rows: 451527


## 2B.1 Continuous Variable Preprocessing

In [6]:
def transform_continuous_variables(df):
    df = add_int_by_extracting_number(
        df,
        source_col="emp_length",
        target_col="emp_length_int",
        special_map={"< 1 year": 0}
    )

    df = add_int_by_extracting_number(
        df,
        source_col="term",
        target_col="term_int"
    )

    df = add_months_since_column(
        df,
        source_col='earliest_cr_line',
        output_col='mths_since_earliest_cr_line',
        reference_date='2017-12-01',
        fmt='MMM-yyyy',
    )

    df = add_months_since_column(
        df,
        source_col='issue_d',
        output_col='mths_since_issue_d',
        reference_date='2017-12-01',
        fmt='MMM-yyyy',
    )

    return df

train_df = transform_continuous_variables(train_df)
test_df  = transform_continuous_variables(test_df)

In [7]:
train_df.select(
    'emp_length', 'emp_length_int',
    'term', 'term_int',
    'earliest_cr_line', 'mths_since_earliest_cr_line',
    'issue_d', 'mths_since_issue_d'
).show(5, truncate=False)

test_df.select(
    'emp_length', 'emp_length_int',
    'term', 'term_int',
    'earliest_cr_line', 'mths_since_earliest_cr_line',
    'issue_d', 'mths_since_issue_d'
).show(5, truncate=False)

+----------+--------------+----------+--------+----------------+---------------------------+--------+------------------+
|emp_length|emp_length_int|term      |term_int|earliest_cr_line|mths_since_earliest_cr_line|issue_d |mths_since_issue_d|
+----------+--------------+----------+--------+----------------+---------------------------+--------+------------------+
|5 years   |5             | 36 months|36      |Jun-1995        |270                        |Feb-2015|34                |
|6 years   |6             | 36 months|36      |Aug-2001        |196                        |Nov-2016|13                |
|8 years   |8             | 36 months|36      |Oct-1998        |230                        |Jun-2015|30                |
|1 year    |1             | 36 months|36      |Jul-1986        |377                        |May-2015|31                |
|6 years   |6             | 60 months|60      |Jul-1989        |341                        |Mar-2015|33                |
+----------+--------------+-----

In [8]:
check_cols = [
    "emp_length", "emp_length_int",
    "term", "term_int",
    "earliest_cr_line", "mths_since_earliest_cr_line",
    "issue_d", "mths_since_issue_d"
]

train_df.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in check_cols
]).show(truncate=False)

test_df.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in check_cols
]).show(truncate=False)

+----------+--------------+----+--------+----------------+---------------------------+-------+------------------+
|emp_length|emp_length_int|term|term_int|earliest_cr_line|mths_since_earliest_cr_line|issue_d|mths_since_issue_d|
+----------+--------------+----+--------+----------------+---------------------------+-------+------------------+
|117604    |117605        |26  |26      |48              |250                        |26     |27                |
+----------+--------------+----+--------+----------------+---------------------------+-------+------------------+

+----------+--------------+----+--------+----------------+---------------------------+-------+------------------+
|emp_length|emp_length_int|term|term_int|earliest_cr_line|mths_since_earliest_cr_line|issue_d|mths_since_issue_d|
+----------+--------------+----+--------+----------------+---------------------------+-------+------------------+
|29336     |29336         |7   |7       |15              |68                         |7

## 2B.2 Discrete Variable Dummy Columns

In [9]:
# discrete_cols = [
#     'grade',
#     'sub_grade',
#     'home_ownership',
#     'verification_status',
#     'purpose',
#     'addr_state',
#     'initial_list_status',
# ]

In [10]:
# dummy_spec = fit_dummy_spec(train_df, discrete_cols, max_categories=80)

# train_df = apply_dummy_columns(train_df, dummy_spec)
# test_df  = apply_dummy_columns(test_df, dummy_spec)

In [11]:
# train_dummy_cols = sorted([c for c in train_df.columns if ':' in c])
# test_dummy_cols = sorted([c for c in test_df.columns if ':' in c])

# print(len(train_dummy_cols), len(test_dummy_cols))
# print(train_dummy_cols == test_dummy_cols)

## 2B.3 Missing Value Treatment

In [12]:
train_missing = get_missing_summary(train_df)
test_missing = get_missing_summary(test_df)

train_missing.show(200, truncate=False)
test_missing.show(200, truncate=False)

+------------------------------------------+-------------+---------------------+
|column                                    |missing_count|missing_ratio        |
+------------------------------------------+-------------+---------------------+
|member_id                                 |1809174      |1.0                  |
|orig_projected_additional_accrued_interest|1802255      |0.996175602788897    |
|hardship_payoff_balance_amount            |1800441      |0.9951729352732241   |
|hardship_last_payment_amount              |1800441      |0.9951729352732241   |
|hardship_loan_status                      |1800437      |0.995170724319496    |
|hardship_dpd                              |1800436      |0.995170171581064    |
|hardship_start_date                       |1800435      |0.995169618842632    |
|hardship_length                           |1800433      |0.9951685133657681   |
|hardship_end_date                         |1800432      |0.995167960627336    |
|payment_plan_start_date    

In [13]:
train_df = fill_null_with_column(train_df, "total_rev_hi_lim", "funded_amnt")
test_df  = fill_null_with_column(test_df,  "total_rev_hi_lim", "funded_amnt")

In [14]:
annual_inc_mean = fit_mean_imputer(train_df, "annual_inc")

train_df = apply_mean_imputer(train_df, "annual_inc", annual_inc_mean)
test_df  = apply_mean_imputer(test_df,  "annual_inc", annual_inc_mean)

In [15]:
zero_fill_cols = [
    "mths_since_earliest_cr_line",
    "acc_now_delinq",
    "total_acc",
    "pub_rec",
    "open_acc",
    "inq_last_6mths",
    "delinq_2yrs",
    "emp_length_int",
]

train_df = fill_null_with_zero(train_df, zero_fill_cols)
test_df  = fill_null_with_zero(test_df,  zero_fill_cols)

In [16]:
check_cols = [
    "total_rev_hi_lim",
    "funded_amnt",
    "annual_inc",
    "mths_since_earliest_cr_line",
    "acc_now_delinq",
    "total_acc",
    "pub_rec",
    "open_acc",
    "inq_last_6mths",
    "delinq_2yrs",
    "emp_length_int",
]

check_null_count(train_df, check_cols, "train_df after imputation")
check_null_count(test_df, check_cols, "test_df after imputation")

=== null check: train_df after imputation ===
+----------------+-----------+----------+---------------------------+--------------+---------+-------+--------+--------------+-----------+--------------+
|total_rev_hi_lim|funded_amnt|annual_inc|mths_since_earliest_cr_line|acc_now_delinq|total_acc|pub_rec|open_acc|inq_last_6mths|delinq_2yrs|emp_length_int|
+----------------+-----------+----------+---------------------------+--------------+---------+-------+--------+--------------+-----------+--------------+
|26              |26         |0         |0                          |0             |0        |0      |0       |0             |0          |0             |
+----------------+-----------+----------+---------------------------+--------------+---------+-------+--------+--------------+-----------+--------------+

=== null check: test_df after imputation ===
+----------------+-----------+----------+---------------------------+--------------+---------+-------+--------+--------------+-----------+

## 2B.4 Target Encoding (Good/Bad)

In [17]:
train_df = add_good_bad_target(train_df)
test_df  = add_good_bad_target(test_df)


In [18]:
train_df.groupBy("loan_status", "good_bad") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(50, truncate=False)

test_df.groupBy("loan_status", "good_bad") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(50, truncate=False)

+---------------------------------------------------+--------+------+
|loan_status                                        |good_bad|count |
+---------------------------------------------------+--------+------+
|Fully Paid                                         |1       |861801|
|Current                                            |1       |702875|
|Charged Off                                        |0       |214897|
|Late (31-120 days)                                 |0       |17133 |
|In Grace Period                                    |1       |6731  |
|Late (16-30 days)                                  |1       |3454  |
|Does not meet the credit policy. Status:Fully Paid |1       |1618  |
|Does not meet the credit policy. Status:Charged Off|0       |602   |
|Default                                            |0       |36    |
|NULL                                               |1       |26    |
|Oct-2015                                           |1       |1     |
+-------------------

In [19]:
train_df.groupBy("good_bad").count().orderBy("good_bad").show()
test_df.groupBy("good_bad").count().orderBy("good_bad").show()

+--------+-------+
|good_bad|  count|
+--------+-------+
|       0| 232668|
|       1|1576506|
+--------+-------+

+--------+------+
|good_bad| count|
+--------+------+
|       0| 58158|
|       1|393369|
+--------+------+



## 2B.7 Save Prepared Datasets

In [ ]:
save_parquet(train_df, '/home/jovyan/work/data/parquet/accepted_general_prep_train.parquet')
save_parquet(test_df, '/home/jovyan/work/data/parquet/accepted_general_prep_test.parquet')

print('Prep parquet files saved.')

Saved to /home/jovyan/work/data/parquet/accepted_general_prep_train.parquet
Saved to /home/jovyan/work/data/parquet/accepted_general_prep_test.parquet
Prep parquet files saved.


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 42450)
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/opt/conda/lib/python3.11/socketserver.py", line 755, in __init__
    self.handle()
  File "/opt/conda/lib/python3.11/site-packages/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/opt/conda/lib/python3.11/site-packages/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
                           ^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/pyspark/accumulators.p